In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/digit-recognizer/sample_submission.csv
/kaggle/input/digit-recognizer/train.csv
/kaggle/input/digit-recognizer/test.csv


# `Using Keras Sequential API to create a model that classfies the MNIST Dataset with 99% accuracy on validation set`

- ## Reading the Train data

In [2]:
df = pd.read_csv('../input/digit-recognizer/train.csv')
df.head(5)

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


- ## Separating the labels from dataset

In [3]:
data = df.drop(['label'],axis=1)
labels = df['label']

In [4]:
labels.unique()

array([1, 0, 4, 7, 3, 5, 8, 9, 2, 6])

- ## Data Preparation

In [5]:
data = data.to_numpy()
labels = labels.to_numpy()

In [6]:
data.shape,labels.shape

((42000, 784), (42000,))

In [7]:
from sklearn.model_selection import train_test_split
train_data , val_data , train_labels , val_labels = train_test_split(data,labels,test_size=0.05)

In [8]:
train_data.shape,val_data.shape

((39900, 784), (2100, 784))

In [9]:
train_data = train_data.reshape(39900,28,28,1)
val_data = val_data.reshape(2100,28,28,1)
train_data = train_data/255
val_data = val_data/255

- ## Importing Tensorflow and Keras

In [10]:
import tensorflow as tf
from tensorflow import keras

- ## Callback function for our model

In [11]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs={}):
        if(logs.get('val_accuracy')>0.99):
            print('Reached 99% validation accuracy,stopping training')
            self.model.stop_training = True
callbacks = myCallback()

- ## Model Creation using Sequential API from Keras

In [12]:
model = keras.Sequential([
    keras.layers.Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
    keras.layers.MaxPooling2D(2,2),
    keras.layers.Conv2D(32,(3,3),activation='relu'),
    keras.layers.MaxPooling2D(2,2),
    keras.layers.Flatten(),
    keras.layers.Dense(256,activation='relu'),
    keras.layers.Dense(10,activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 26, 26, 32)        320       
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 13, 13, 32)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 11, 11, 32)        9248      
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 5, 5, 32)          0         
_________________________________________________________________
flatten (Flatten)            (None, 800)               0         
_________________________________________________________________
dense (Dense)                (None, 256)               205056    
_________________________________________________________________
dense_1 (Dense)              (None, 10)                2

- ## Training the model on training data

In [13]:
model.fit(train_data,train_labels,epochs=50,validation_data=(val_data,val_labels),callbacks=[callbacks])

Epoch 1/50
1247/1247 [==============================] - 18s 14ms/step - loss: 0.3683 - accuracy: 0.8880 - val_loss: 0.0840 - val_accuracy: 0.9710
Epoch 2/50
1247/1247 [==============================] - 17s 14ms/step - loss: 0.0603 - accuracy: 0.9800 - val_loss: 0.0417 - val_accuracy: 0.9852
Epoch 3/50
1247/1247 [==============================] - 17s 14ms/step - loss: 0.0362 - accuracy: 0.9886 - val_loss: 0.0419 - val_accuracy: 0.9852
Epoch 4/50
1247/1247 [==============================] - 17s 14ms/step - loss: 0.0236 - accuracy: 0.9928 - val_loss: 0.0477 - val_accuracy: 0.9857
Epoch 5/50
1247/1247 [==============================] - 17s 14ms/step - loss: 0.0189 - accuracy: 0.9936 - val_loss: 0.0597 - val_accuracy: 0.9852
Epoch 6/50
1247/1247 [==============================] - 17s 14ms/step - loss: 0.0139 - accuracy: 0.9955 - val_loss: 0.0392 - val_accuracy: 0.9886
Epoch 7/50
1247/1247 [==============================] - 17s 14ms/step - loss: 0.0110 - accuracy: 0.9965 - val_loss: 0.0657 -

- ## Reading the test dataset

In [14]:
test_data = pd.read_csv('../input/digit-recognizer/test.csv')
test_data

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


- ## Data Preparation for Test set and making predictions on it

In [15]:
test_data = test_data.to_numpy()
test_data = test_data.reshape(28000,28,28,1)
test_data = test_data / 255
predictions = model.predict(test_data)
predictions = np.argmax(predictions,axis=1)
predictions

array([2, 0, 9, ..., 3, 9, 2])

- ## Creating a Pandas DataFrame with our predictions in it

In [16]:
submission = pd.read_csv('../input/digit-recognizer/sample_submission.csv')
submission['Label'] = predictions

- ## Exporting our dataframe to CSV file format

In [17]:
submission.to_csv('mnist_digit_test_predictions.csv',index=False)

# The End
`If you liked the notebook then don't forget to upvote and suggestions are always welcomed.`
`Follow me on Linkedin :` __[Atharva_Dumbre](https://www.linkedin.com/in/atharva-dumbre-208b5716b)__